In [1]:
from os import path, mkdir, getcwd

import numpy as np
from tqdm import tqdm

from efsprapy.mcs1 import MCS1, EXAMPLE_INPUT
from sfeprapy.mcs import InputParser
from fsetools.lib.fse_thermal_radiation_v2 import phi_solver
from sfeprapy.func.xlsx import dict_to_xlsx
from itertools import product
from copy import deepcopy


getcwd()

'C:\\Users\\Yan\\Documents\\GitHub\\EFSPRAPY\\test\\mcs1-br187_84'

In [2]:
# inputs
fp_input = path.join(getcwd(), "br187-084.xlsx")
dir_work = path.join(path.dirname(fp_input))
try:
    mkdir(dir_work)
except:
    pass

assert path.exists(fp_input), f'file does not exist {fp_input}'

In [3]:
Wv = list(np.arange(3, 30, 0.5)) + [30, 33, 36, 39, 42, 45, 48, 51, 54, 57, 60,]
Hv = list(np.arange(3, 30, 0.5)) + [30,]
kwargs = dict()
for W, H in product(Wv, Hv):
    _, _, S, _ = phi_solver(W, H, W * 0.5, H * 0.5, 0, 84, 12.6, None, 1)
    case_name = f"{W:06.3f}-{H:06.3f}-{S:06.3f}"
    kwargs[case_name] = deepcopy(EXAMPLE_INPUT["CASE_1"])
    kwargs[case_name].update(
        dict(
            n_simulations=100_000,
            fire_mode=0,
            room_width=W,
            room_height=H,
            opening_width=W,
            opening_height=H,
            receiver_separation=S,
            ftp_chf=13.3e3,
            ftp_index=1.7,
            ftp_target=9094,
            receiver_ignition_temperature=-1,
            safir_input_file_s=None,
        )
    )

In [ ]:
dict_to_xlsx({k_: InputParser.flatten_dict(v_) for k_, v_ in kwargs.items()}, fp_input)
mcs = MCS1()
mcs.set_inputs_file_path(fp_input)
pbar = tqdm()
mcs.run(
    10,
    lambda _: pbar.update(1),
    lambda _: setattr(pbar, "total", _),
    save=True,
    save_archive=False,
    concurrency_strategy=1,
)
pbar.close()

 20%|█▉        | 700/3575 [22:58<42:40,  1.12it/s]  

In [ ]:
from helper_funcs import plot_contour, read_output_and_gen_xyz


d = read_output_and_gen_xyz(fp_input, 30)
xv, yv = np.sort(np.unique(d[:, 0])), np.sort(np.unique(d[:, 1]))
xx, yy = np.meshgrid(xv, yv)
zz = np.zeros_like(xx)

for x, y, z in d:
    try:
        zz[int(np.argwhere(yv==y)), int(np.argwhere(xv==x))] = z
    except:
        print(int(np.argwhere(yv==y)), int(np.argwhere(xv==x)))
        raise ValueError()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt


plt.style.use('seaborn-v0_8-talk')

fig, ax = plt.subplots(figsize=(5.90551, 5.90551), dpi=100)  # (7.48031, 5.90551)

cf = plot_contour(
    ax=ax, xx=xx, yy=yy, zz=zz*100, xlabel=r'Width [m]', ylabel='Height [m]',
    levels=[0.1, 1, 2, 4, 6, 8, 10, 20, 30, 40, 50, ], clabel_fmt=lambda x: f'{x:.0f} %',
    xlim=(3, 60), 
    xticks=list(range(3, 16, 3)) + list(range(15, 101, 5)), 
    xticklabels=None,
    ylim=(3, 30), 
    yticks=list(range(3, 16, 3)) + list(range(15, 101, 5)), 
    legend_visible=False,
)

fig.savefig(
    path.join(path.dirname(fp_input), f'{path.splitext(path.basename(fp_input))[0]}.png'), 
    dpi=600,
    bbox_inches='tight', 
    pad_inches=0.02, 
    transparent=True
)